In [ ]:
#@title 📥 Download & Install Latest Blender (Clean Install)
import urllib.request
import re
import os
import sys
import shutil

print("0. Cleaning up old Blender files from previous attempts...")
for item in os.listdir("."):
    if os.path.isdir(item) and item.startswith("blender-"):
        shutil.rmtree(item)

print("1. Installing virtual display system & graphics libraries...")
os.system("apt-get update && apt-get install -y x11-utils libsdl2-dev libglu1-mesa-dev freeglut3-dev mesa-utils xvfb libxkbcommon-x11-0 libxrender1 libxxf86vm1 libxi6 libsm6 > /dev/null 2>&1")

print("2. Scanning Blender servers for the newest daily build...")
builder_url = "https://builder.blender.org/download/daily/"
user_agent_headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}

try:
    req = urllib.request.Request(builder_url, headers=user_agent_headers)
    html = urllib.request.urlopen(req).read().decode('utf-8')

    # Grab EVERY Linux tar.xz link on the page
    links = re.findall(r'href="([^"]+linux[^"]+\.tar\.xz)"', html)

    if not links:
        print("❌ Could not find ANY Linux downloads on the page.")
    else:
        target_link = None

        # 1st Priority: Look for 5.1 specifically
        for link in links:
            if "5.1" in link:
                target_link = link
                break

        # 2nd Priority: Look for "main" branch (the newest experimental Alpha)
        if not target_link:
            for link in links:
                if "main" in link:
                    target_link = link
                    break

        # 3rd Priority: Just grab the first Linux link available
        if not target_link:
            target_link = links[0]

        # Format the URL correctly
        if target_link.startswith('/'):
            download_url = "https://builder.blender.org" + target_link
        else:
            download_url = target_link

        tar_filename = download_url.split("/")[-1]
        print(f"3. Found Newest Build: {tar_filename}")

        print("4. Downloading...")
        download_req = urllib.request.Request(download_url, headers=user_agent_headers)
        with urllib.request.urlopen(download_req) as response:
            total_size = int(response.headers.get('content-length', 0))
            block_size = 1024 * 1024 # 1 MB blocks
            downloaded = 0

            with open(tar_filename, 'wb') as out_file:
                while True:
                    buffer = response.read(block_size)
                    if not buffer:
                        break
                    downloaded += len(buffer)
                    out_file.write(buffer)
                    if total_size:
                        percent = int(downloaded * 100 / total_size)
                        sys.stdout.write(f"\r   Progress: {percent}% ({downloaded // (1024*1024)}MB / {total_size // (1024*1024)}MB)")
                        sys.stdout.flush()

        print("\n5. Extracting files (this takes about 15 seconds)...")
        os.system(f"tar -xf {tar_filename}")

        print("6. Locating exact program path...")
        blender_executable_path = None
        for root, dirs, files in os.walk("."):
            if "/." in root:
                continue
            if "blender" in files:
                temp_path = os.path.join(root, "blender")
                if os.path.isfile(temp_path) and os.access(temp_path, os.X_OK):
                    blender_executable_path = temp_path
                    break

        if blender_executable_path:
            with open("blender_path.txt", "w") as f:
                f.write(blender_executable_path)
            print(f"\n🎉 Success! Blender located at: {blender_executable_path}")
            os.remove(tar_filename)
        else:
            print("\n❌ Extraction failed or could not locate the 'blender' file inside.")

except Exception as e:
    print(f"❌ Error setting up Blender: {e}")

In [ ]:
#@title 🎛| Ultimate Render Farm Controller (Cycles Turbo Mode)

#@markdown ### 📂 File Settings
#@markdown *Type the name of the file (e.g., "planet" or "planet.blend"). The script handles the rest!*
project_name = "planet" #@param {type:"string"}
output_video_name = "Space_Render" #@param {type:"string"}

#@markdown ---
#@markdown ### 💾 Storage & Recovery Settings
Save_to_Google_Drive = True #@param {type:"boolean"}
#@markdown *If Colab crashed, check this box. Blender will SKIP already rendered frames and pick up exactly where it left off!*
Resume_Interrupted_Render = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### 🎥 Video Resolution
width = 1920 #@param {type:"integer"}
height = 1080 #@param {type:"integer"}
fps = 24 #@param {type:"integer"}

#@markdown ---
#@markdown ### 💎 Render Quality (Cycles)
#@markdown *32 is great for testing. 128 is highly recommended for final, noise-free space animations.*
render_samples = 128 #@param [4, 8, 16, 32, 64, 128, 256, 512, 1024, 2048, 4096] {type:"raw"}

#@markdown ---
#@markdown ### ⏱️ Frame Range
start_frame = 1 #@param {type:"integer"}
end_frame = 250 #@param {type:"integer"}

#@markdown ---
#@markdown ### ⚠️ Advanced Settings
#@markdown *Check this ONLY if your file relies on custom Python scripts (like custom drivers) to animate.*
allow_embedded_scripts = False #@param {type:"boolean"}

import os
import shutil

# 1. HANDLE GOOGLE DRIVE MOUNTING
if Save_to_Google_Drive:
    from google.colab import drive
    drive.mount('/content/drive')
    output_dir = "/content/drive/MyDrive/BlenderRender"
    os.makedirs(output_dir, exist_ok=True)
    print("\n✅ Google Drive connected! Frames will be saved safely to the cloud.")
else:
    output_dir = "/content"

# 2. SMART FILE LOCATOR
if project_name.endswith(".blend"):
    blend_name = project_name
else:
    blend_name = f"{project_name}.blend"

if os.path.exists(f"/content/{blend_name}"):
    input_path = f"/content/{blend_name}"
    search_dir = "/content"
elif Save_to_Google_Drive and os.path.exists(f"/content/drive/MyDrive/BlenderRender/{blend_name}"):
    input_path = f"/content/drive/MyDrive/BlenderRender/{blend_name}"
    search_dir = "/content/drive/MyDrive/BlenderRender"
else:
    raise FileNotFoundError(f"❌ Could not find '{blend_name}' in the Colab Files tab OR Google Drive!")

# 3. SETUP FOLDER PATHS
final_mp4_path = f"{output_dir}/{output_video_name}_{start_frame:04d}-{end_frame:04d}.mp4"

# Create a dedicated folder just for this video's frames
frames_dir = f"{output_dir}/{output_video_name}_frames"
os.makedirs(frames_dir, exist_ok=True)

# If NOT resuming, clean out the old frames so we get a fresh start
if not Resume_Interrupted_Render:
    for f in os.listdir(frames_dir):
        os.remove(os.path.join(frames_dir, f))
else:
    print("\n♻️ RESUME MODE ACTIVE: Blender will skip existing frames and only render missing ones!")

if os.path.exists("blender_path.txt"):
    with open("blender_path.txt", "r") as f:
        blender_bin = f.read().strip()
else:
    blender_bin = "./blender-5.1-portable/blender"

# Python string conversion for the overwrite flag
overwrite_val = "False" if Resume_Interrupted_Render else "True"

gpu_script = f"""
import bpy
import os

print("\\n--- STARTING SUPER-OPTIMIZED BLENDER SETUP ---")

# 1. FORCE CYCLES ENGINE
scene = bpy.context.scene
scene.render.engine = 'CYCLES'

scene.render.resolution_x = {width}
scene.render.resolution_y = {height}

# 🚨 CRITICAL PERFORMANCE OPTIMIZATIONS 🚨
scene.render.use_compositing = False
scene.render.use_sequencer = False
scene.render.use_motion_blur = False

# PERSISTENT DATA: Keeps the scene in VRAM between frames.
scene.render.use_persistent_data = True

# 🚨 THE TILE KILLER WORKAROUND 🚨
# We force the tile size to 4096. Since your resolution is 1920x1080,
# Blender is forced to render the entire image as one single tile (1/1 Tiles)!
scene.cycles.use_auto_tile = True
scene.cycles.tile_size = 4096

# CRASH-PROOF FORMAT SETTINGS
try:
    scene.render.image_settings.file_format = 'PNG'
    scene.render.image_settings.color_mode = 'RGBA'
except Exception as e:
    print(f"[NOTE] Format override bypassed safely: {{e}}")

scene.render.use_overwrite = {overwrite_val}

print("Checking for missing textures...")
for img in bpy.data.images:
    if getattr(img, 'filepath', '') and img.source != 'GENERATED':
        filename = bpy.path.basename(img.filepath)
        if filename:
            new_path = os.path.join("{search_dir}", filename)
            img.filepath = new_path
            try:
                img.reload()
                print(f"  [SUCCESS] Relinked: {{filename}}")
            except Exception:
                print(f"  [WARNING] Could not load: {{filename}}")

# 2. ENABLE NVIDIA GPU
try:
    scene.cycles.device = 'GPU'
    prefs = bpy.context.preferences
    cycles_prefs = prefs.addons['cycles'].preferences
    cycles_prefs.compute_device_type = 'OPTIX'
    cycles_prefs.get_devices()

    for device in cycles_prefs.devices:
        if device.type == 'OPTIX':
            device.use = True
        else:
            device.use = False
except Exception as e:
    print("GPU initialization note:", e)

# SPEED & QUALITY OPTIMIZATIONS
scene.cycles.samples = {render_samples}
scene.cycles.use_denoising = False

# Light does not bounce in a vacuum. Disable diffuse/glossy to drastically speed up renders.
scene.cycles.max_bounces = 1
scene.cycles.diffuse_bounces = 1
scene.cycles.glossy_bounces = 1
scene.cycles.transmission_bounces = 0
scene.cycles.volume_bounces = 0
scene.cycles.transparent_max_bounces = 8
scene.cycles.caustics_reflective = False
scene.cycles.caustics_refractive = False

print("--- SETUP COMPLETE, STARTING RENDER ---\\n")
"""

with open("setup_maxspeed.py", "w") as file:
    file.write(gpu_script)

security_flag = "-y" if allow_embedded_scripts else ""

print("\n1. Rendering PNG Sequence via CYCLES...")
!{blender_bin} -b "{input_path}" {security_flag} -P "setup_maxspeed.py" -F PNG -o "{frames_dir}/frame_" -s {start_frame} -e {end_frame} -a

print("\n2. Stitching frames into MP4 video...")
# Uses Colab's native FFMPEG to compile the sequence.
!ffmpeg -y -framerate {fps} -start_number {start_frame} -i "{frames_dir}/frame_%04d.png" -c:v libx264 -preset fast -pix_fmt yuv420p "{final_mp4_path}" -loglevel warning

# We only delete the images if we successfully finished the video and ARE NOT in resume mode
if not Resume_Interrupted_Render:
    print("\n3. Cleaning up temporary image files...")
    shutil.rmtree(frames_dir)
else:
    print("\n3. Keeping temporary frames just in case (Resume Mode is on).")

print("\n✅ RENDER COMPLETE!")
print(f"Your video has been saved to: {final_mp4_path}")